<a href="https://colab.research.google.com/github/hilalbht/colab.github.io/blob/main/midtermEAproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:

import math
import random
import numpy as np
import sys

DOSYA_ADI = 'berlin52 (1).tsp'   # Örnekler: berlin52.tsp, att48.tsp, a280.tsp, att532.tsp
POP_BOYUTU = 100
MAX_GENERATION = 100
NO_IMPROVEMENT_LIMIT = 5
MUTASYON_ORANI = 0.3


                     #INITIALIZATION

# 1)veri okuma
def veriyi_oku(dosya):
    sehir_coords = {}
    basla_oku = False
    try:
        with open(dosya, 'r') as f:
            for satir in f:
                satir = satir.strip()
                if satir == 'NODE_COORD_SECTION':
                    basla_oku = True
                    continue
                if satir == 'EOF':
                    break
                if not basla_oku:
                    continue
                if satir == '':
                    continue
                parcalar = satir.split()
                if len(parcalar) >= 3:
                    try:
                        idx = int(parcalar[0]) - 1
                        x = float(parcalar[1])
                        y = float(parcalar[2])
                        sehir_coords[idx] = (x, y)
                    except ValueError:

                        pass
    except FileNotFoundError:
        print(f"HATA: {dosya} dosyası bulunamadı. Lütfen dosya adını ve konumunu kontrol et.")
        return {}, 0

    return sehir_coords, len(sehir_coords)

SEHIRLER, SEHIR_SAYISI = veriyi_oku(DOSYA_ADI)
if SEHIR_SAYISI == 0:
    sys.exit("Şehir sayısı 0. Program sonlandırıldı.")

# kromozom olusturma/fitness
def mesafe_hesapla(koordinat1, koordinat2):
    x1, y1 = koordinat1
    x2, y2 = koordinat2
    return math.hypot(x2 - x1, y2 - y1)

def rota_fitness_hesapla(kromozom, sehir_koordinatlari):
    toplam_mesafe = 0.0
    N = len(kromozom)
    for i in range(N):
        a = kromozom[i]
        b = kromozom[(i + 1) % N]
        coord_a = sehir_koordinatlari[a]
        coord_b = sehir_koordinatlari[b]
        toplam_mesafe += mesafe_hesapla(coord_a, coord_b)
    return toplam_mesafe

# 3) rastgele populasyon olusturma
def ilk_populasyonu_kur(sehir_sayisi, pop_boyutu=100):
    populasyon = []
    sehir_indeksleri = list(range(sehir_sayisi))
    for _ in range(pop_boyutu):
        yeni = sehir_indeksleri[:]
        random.shuffle(yeni)
        populasyon.append(yeni)
    return populasyon

                          #ITERATIONS

# 4- a -i) sıra tabanlı secim (rank)
def sira_tabanli_secim(populasyon_degerlendirilmis, kac_tane_sec=1):
    pop_sorted = sorted(populasyon_degerlendirilmis, key=lambda x: x[1])
    pop_boyutu = len(pop_sorted)
    puanlar = [pop_boyutu - i for i in range(pop_boyutu)]
    toplam = sum(puanlar)
    olasilik = [p / toplam for p in puanlar]
    secilen_idx = np.random.choice(pop_boyutu, size=kac_tane_sec, p=olasilik, replace=True)
    secilenler = [pop_sorted[i][0] for i in secilen_idx]
    return secilenler

# 4 -a -ii) rulet tekerlegi secimi
def rulet_tekerlegi_secimi(populasyon_degerlendirilmis, kac_tane_sec=1):
    fitnessler = [f for _, f in populasyon_degerlendirilmis]
    eps = 1e-8
    inv = [1.0 / (f + eps) for f in fitnessler]
    toplam = sum(inv)
    probs = [v / toplam for v in inv]
    pop_boyutu = len(populasyon_degerlendirilmis)
    secilen_idx = np.random.choice(pop_boyutu, size=kac_tane_sec, p=probs, replace=True)
    secilenler = [populasyon_degerlendirilmis[i][0] for i in secilen_idx]
    return secilenler

#  ebeveyn secimi ( 50 rank, 50 rulet)
def ebeveyn_sec(pop_degerlendirilmis, kalan_cozum_sayisi=99):

    yarisi = kalan_cozum_sayisi // 2
    diger = kalan_cozum_sayisi - yarisi
    rankli = sira_tabanli_secim(pop_degerlendirilmis, yarisi)
    ruletli = rulet_tekerlegi_secimi(pop_degerlendirilmis, diger)
    ebeveynler = rankli + ruletli
    random.shuffle(ebeveynler)
    return ebeveynler

# 4- b) cycle crossover (CX)
def cycle_crossover(parent1, parent2):
    n = len(parent1)
    child1 = [-1] * n
    child2 = [-1] * n
    idx = 0
    while -1 in child1:
        for i in range(n):
            if child1[i] == -1:
                idx = i
                break

        start = idx
        val = parent1[start]
        cur = start
        cycle_indices = []
        while True:
            cycle_indices.append(cur)
            val = parent1[cur]
            cur = parent2.index(val)
            if cur == start:
                break

        for ci in cycle_indices:
            child1[ci] = parent1[ci]
            child2[ci] = parent2[ci]

    for i in range(n):
        if child1[i] == -1:
            child1[i] = parent2[i]
        if child2[i] == -1:
            child2[i] = parent1[i]
    return child1, child2

def dongu_caprazlamasi_cx(p1, p2):
    return cycle_crossover(p1, p2)

# 4 -c- i)araya ekleme mutasyonu
def araya_ekleme_mutasyonu(kromozom, mutasyon_orani=0.01):
    if random.random() > mutasyon_orani:
        return kromozom[:]
    k = kromozom[:]
    N = len(k)
    i, j = random.sample(range(N), 2)
    gene = k.pop(i)
    k.insert(j, gene)
    return k

# 4 -c- ii)rastgele kaydırma mutasyonu

def rastgele_kaydirma_mutasyonu(kromozom, mutasyon_orani=0.01):
    if random.random() > mutasyon_orani:
        return kromozom[:]
    k = kromozom[:]
    N = len(k)
    a, b = sorted(random.sample(range(N), 2))
    segment = k[a:b+1]
    L = len(segment)
    if L <= 1:
        return k
    offset = random.randint(1, L-1)
    rotated = segment[offset:] + segment[:offset]
    k[a:b+1] = rotated
    return k
                      #  TERMINATION


# 5)genetik algoritma dongusu ve sonlandırma

def run_genetic_algorithm():

    pop_kromozom = ilk_populasyonu_kur(SEHIR_SAYISI, POP_BOYUTU)
    populasyon = [(k, rota_fitness_hesapla(k, SEHIRLER)) for k in pop_kromozom]
    populasyon.sort(key=lambda x: x[1])

    en_iyi_ilk_krom, en_iyi_ilk_f = populasyon[0]
    print(f"İlk nesil- en iyi fitness (elitizm ile korunacak) : {en_iyi_ilk_f:.4f}")    #elitizm
    print(f"İlk nesil- en iyi kromozom (elit) : {en_iyi_ilk_krom}")

    best_history = []
    best_fitness = populasyon[0][1]
    no_improve_count = 0

    print("Genetik algoritma basladı")
    for gen in range(1, MAX_GENERATION + 1):

        populasyon.sort(key=lambda x: x[1])
        en_iyi_krom, en_iyi_f = populasyon[0]
        best_history.append(en_iyi_f)

        print(f"Nesil {gen:3d} -> En iyi fitness degeri: {en_iyi_f:.4f}")

        if en_iyi_f < best_fitness - 1e-12:
            best_fitness = en_iyi_f
            no_improve_count = 0
        else:
            no_improve_count += 1

        if no_improve_count >= NO_IMPROVEMENT_LIMIT:
            print(f" En iyi çözüm {NO_IMPROVEMENT_LIMIT} ardışık nesilde değişmedi. Durduruldu.")
            break

        if gen >= MAX_GENERATION:
            print(f"Maksimum nesil ({MAX_GENERATION}) tamamlandı.")
            break

        # elitizm - en iyi bireyi taşı
        yeni_pop_kromozom = [en_iyi_krom[:]]

        # ebeveyn seçimi (kalan POP_BOYUTU - 1 için)
        ebeveynler = ebeveyn_sec(populasyon, kalan_cozum_sayisi=POP_BOYUTU - 1)

        # çaprazlama - ebeveyn çiftlerinden çocuklar üret
        cocuklar = []
        if len(ebeveynler) % 2 == 1:
            ebeveynler.append(random.choice(ebeveynler))
        for i in range(0, len(ebeveynler), 2):
            p1 = ebeveynler[i]
            p2 = ebeveynler[i+1]
            c1, c2 = dongu_caprazlamasi_cx(p1, p2)
            cocuklar.append(c1)
            if len(cocuklar) < POP_BOYUTU - 1:
                cocuklar.append(c2)
            if len(cocuklar) >= POP_BOYUTU - 1:
                break


        random.shuffle(cocuklar)
        yarisi = len(cocuklar) // 2
        yeni_cocuklar = []
        for idx, chrom in enumerate(cocuklar):
            if idx < yarisi:

                yeni = araya_ekleme_mutasyonu(chrom, mutasyon_orani=MUTASYON_ORANI)
            else:
                yeni = rastgele_kaydirma_mutasyonu(chrom, mutasyon_orani=MUTASYON_ORANI)
            yeni_cocuklar.append(yeni)

        #  yeni popülasyon oluştur ((1)elit + (99)yavrular
        for c in yeni_cocuklar:
            if len(novi := yeni_pop_kromozom) >= POP_BOYUTU:
                break
            yeni_pop_kromozom.append(c)


        while len(yeni_pop_kromozom) < POP_BOYUTU:
            yeni_pop_kromozom.append(random.choice(ilk_populasyonu_kur(SEHIR_SAYISI, 1)))

        populasyon = [(k, rota_fitness_hesapla(k, SEHIRLER)) for k in yeni_pop_kromozom]
        populasyon.sort(key=lambda x: x[1])

   #sonuc
    populasyon.sort(key=lambda x:x[1])
    en_iyi_krom, en_iyi_f = populasyon[0]
    print("GENETİK ALGORİTMA TAMAMLANDI")
    print(f"En iyi rota uzunluğu: {en_iyi_f:.4f}")
    print(f"En iyi rota: {en_iyi_krom}")
    return en_iyi_krom, en_iyi_f, best_history

if __name__ == "__main__":
    en_krom, en_f, history = run_genetic_algorithm()


İlk nesil- en iyi fitness (elitizm ile korunacak) : 24956.7702
İlk nesil- en iyi kromozom (elit) : [25, 13, 12, 40, 48, 35, 8, 22, 41, 15, 49, 1, 6, 45, 27, 34, 16, 31, 50, 46, 26, 5, 29, 51, 14, 36, 44, 2, 0, 39, 33, 11, 32, 47, 23, 7, 30, 38, 24, 28, 20, 18, 21, 17, 42, 3, 19, 4, 10, 43, 37, 9]
Genetik algoritma basladı
Nesil   1 -> En iyi fitness degeri: 24956.7702
Nesil   2 -> En iyi fitness degeri: 24956.7702
Nesil   3 -> En iyi fitness degeri: 24956.7702
Nesil   4 -> En iyi fitness degeri: 24956.7702
Nesil   5 -> En iyi fitness degeri: 24956.7702
 En iyi çözüm 5 ardışık nesilde değişmedi. Durduruldu.
GENETİK ALGORİTMA TAMAMLANDI
En iyi rota uzunluğu: 24956.7702
En iyi rota: [25, 13, 12, 40, 48, 35, 8, 22, 41, 15, 49, 1, 6, 45, 27, 34, 16, 31, 50, 46, 26, 5, 29, 51, 14, 36, 44, 2, 0, 39, 33, 11, 32, 47, 23, 7, 30, 38, 24, 28, 20, 18, 21, 17, 42, 3, 19, 4, 10, 43, 37, 9]
